# Pórtico plano de un vano: método matricial de rigidez

**Fuente:** [Láminas del curso](https://view.genially.com/684f80b53c7df3efd8dc19f4), 250–253.

Pórtico 253: B=1, C=2, A=3, D=4. Compruebe la simetría de desplazamientos y reacciones.

Unidades: **kN, m, rad**. Python es complementario. Puede realizar el procedimiento con Excel y cálculo a mano.

## Antes de ejecutar
1. Dibuje la estructura y numere los GDL. ¿Cuáles pertenecen a n y a a?
2. Escriba las cargas aplicadas y las fuerzas de empotramiento con sus signos.
3. Prediga un desplazamiento y una reacción.
4. Explique por qué no se invierte K antes de imponer apoyos.

Abra este archivo en Colab y seleccione **Entorno de ejecución, Ejecutar todas**. Edite solo las propiedades positivas y las cargas señaladas. Geometría, topología, apoyos y orden de GDL son fijos.

## Procedimiento en seis pasos
1. Identifique la estructura, numere los nudos y determine la orientación.
2. Calcule las matrices de los miembros en coordenadas generales.
3. Ensamble K y reordene separando fuerzas de nudos libres y reacciones.
4. Particione K y calcule los desplazamientos.
5. Calcule las reacciones y verifique el equilibrio general.
6. Calcule las fuerzas internas con las matrices individuales y verifique el equilibrio de nudos.


## 1. Datos y unidades

Fuente diap.253: A y D empotrados, altura 3 m, luz 6 m y w=25 kN/m hacia abajo en BC.

Nudos de la fuente: 1=B, 2=C, 3=A, 4=D. Orientaciones: 3→1, 1→2, 4→2.

Secciones rectangulares: viga b×h=300×500 mm; columnas 300×300 mm. E=19 kN/mm².

A e I se derivan de A=b·h e I=b·h³/12: viga A=0.15 m², I=0.003125 m⁴; columnas A=0.09 m², I=0.000675 m⁴.

Orden elemental del curso: [ui,vi,θi,uj,vj,θj]. Se incluye deformación axial y flexión Euler–Bernoulli.

w se introduce positivo en −y local. En la viga horizontal 1→2: w=25 kN/m hacia abajo. Los giros y momentos positivos son antihorarios.

Calcule Δx=xj−xi, Δy=yj−yi, L=√(Δx²+Δy²), c=Δx/L y s=Δy/L. T es adimensional.

n identifica los GDL libres y a los de apoyos. δa es conocido. F^F son las fuerzas de empotramiento. La reacción es Fa menos la carga aplicada directamente en el apoyo.

Orden global adoptado: [u1,v1,θ1,u2,v2,θ2,u3,v3,θ3,u4,v4,θ4]. La fuente sí fija nudos y orden elemental.


In [ ]:
import json, copy, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.set_printoptions(precision=6, suppress=True, linewidth=130)

# Datos originales de topología fija. Cambie solo propiedades positivas y cargas.
caso = json.loads(r'''{
  "id": "portico",
  "kind": "frame",
  "title": "Pórtico plano de un vano",
  "example": "Problema de pórtico",
  "slides": "250–253",
  "source_url": "https://view.genially.com/684f80b53c7df3efd8dc19f4",
  "nodes": [
    {
      "id": "1",
      "x": 0.0,
      "y": 3.0,
      "label": "1 = B"
    },
    {
      "id": "2",
      "x": 6.0,
      "y": 3.0,
      "label": "2 = C"
    },
    {
      "id": "3",
      "x": 0.0,
      "y": 0.0,
      "label": "3 = A"
    },
    {
      "id": "4",
      "x": 6.0,
      "y": 0.0,
      "label": "4 = D"
    }
  ],
  "elements": [
    {
      "id": "31",
      "i": "3",
      "j": "1",
      "E_parameter": "E_material",
      "A": 0.09,
      "I": 0.0006749999999999998,
      "w": -0.0
    },
    {
      "id": "12",
      "i": "1",
      "j": "2",
      "E_parameter": "E_material",
      "A": 0.15,
      "I": 0.0031249999999999997,
      "w": 25.0
    },
    {
      "id": "42",
      "i": "4",
      "j": "2",
      "E_parameter": "E_material",
      "A": 0.09,
      "I": 0.0006749999999999998,
      "w": -0.0
    }
  ],
  "parameters": [
    {
      "key": "E_material",
      "value": 19000000.0,
      "unit": "kN/m²",
      "description": "E de la fuente: 19 kN/mm².",
      "stiffness_parameter": true
    }
  ],
  "dofs": [
    {
      "node": "1",
      "component": "u",
      "restrained": false,
      "load": 0,
      "label": "u1"
    },
    {
      "node": "1",
      "component": "v",
      "restrained": false,
      "load": 0,
      "label": "v1"
    },
    {
      "node": "1",
      "component": "theta",
      "restrained": false,
      "load": 0,
      "label": "θ1"
    },
    {
      "node": "2",
      "component": "u",
      "restrained": false,
      "load": 0,
      "label": "u2"
    },
    {
      "node": "2",
      "component": "v",
      "restrained": false,
      "load": 0,
      "label": "v2"
    },
    {
      "node": "2",
      "component": "theta",
      "restrained": false,
      "load": 0,
      "label": "θ2"
    },
    {
      "node": "3",
      "component": "u",
      "restrained": true,
      "load": 0,
      "label": "u3"
    },
    {
      "node": "3",
      "component": "v",
      "restrained": true,
      "load": 0,
      "label": "v3"
    },
    {
      "node": "3",
      "component": "theta",
      "restrained": true,
      "load": 0,
      "label": "θ3"
    },
    {
      "node": "4",
      "component": "u",
      "restrained": true,
      "load": 0,
      "label": "u4"
    },
    {
      "node": "4",
      "component": "v",
      "restrained": true,
      "load": 0,
      "label": "v4"
    },
    {
      "node": "4",
      "component": "theta",
      "restrained": true,
      "load": 0,
      "label": "θ4"
    }
  ],
  "convention": "Orden global adoptado: [u1,v1,θ1,u2,v2,θ2,u3,v3,θ3,u4,v4,θ4]. La fuente sí fija nudos y orden elemental.",
  "notes": [
    "Fuente diap.253: A y D empotrados, altura 3 m, luz 6 m y q=25 kN/m hacia abajo en BC.",
    "Nudos de la fuente: 1=B, 2=C, 3=A, 4=D. Orientaciones: 3→1, 1→2, 4→2.",
    "Secciones rectangulares: viga b×h=300×500 mm; columnas 300×300 mm. E=19 kN/mm².",
    "A e I se derivan de A=b·h e I=b·h³/12: viga A=0.15 m², I=0.003125 m⁴; columnas A=0.09 m², I=0.000675 m⁴.",
    "Orden elemental del curso: [ui,vi,θi,uj,vj,θj]. Se incluye deformación axial y flexión Euler–Bernoulli.",
    "q se introduce en +y local. En el elemento horizontal 1→2: q=−25 kN/m. Los giros y momentos positivos son antihorarios.",
    "Calcule Δx=xj−xi, Δy=yj−yi, L=√(Δx²+Δy²), c=Δx/L y s=Δy/L. T es adimensional.",
    "n identifica los GDL libres y a los de apoyos. δa es conocido. Fᶠ es el negativo de la carga equivalente."
  ],
  "result_note": "Pórtico 253: B=1, C=2, A=3, D=4. Compruebe la simetría de desplazamientos y reacciones.",
  "source_results": null,
  "conventions": [],
  "load_convention": "w > 0 actúa en −Ȳ local: hacia abajo en las vigas horizontales. Las cargas nodales conservan el signo de los ejes globales."
}''')
# Identifica los datos de referencia para no aplicar sus controles a una variante.
hash_original = 'faa9c5562d3f5a4b43957a8f7d662925baa052ace1d4b2d83587fde755657d6b'


## 2. Esquema y grados de libertad

Compruebe nudos, orientación de elementos y restricciones con la fuente. La figura no es la deformada.

In [ ]:
import base64, io
# Redibujo del enunciado y, en la viga, su reducción estática.
fig, ax = plt.subplots(figsize=(10,5))
ax.imshow(plt.imread(io.BytesIO(base64.b64decode("...")),format="png")); ax.axis("off"); plt.show()

nudos = {n['id']:n for n in caso['nodes']}
fig, ax = plt.subplots(figsize=(9,4))
for e in caso['elements']:
    ni,nj=nudos[e['i']],nudos[e['j']]
    ax.plot([ni['x'],nj['x']],[ni['y'],nj['y']],'-o',color='#24558A',lw=2)
    ax.text((ni['x']+nj['x'])/2+.05,(ni['y']+nj['y'])/2+.1,e['id'])
for nd in caso['nodes']:
    ax.text(nd['x']+.08,nd['y']-.15,nd['label'])
    restr=[d['component'] for d in caso['dofs'] if d['node']==nd['id'] and d['restrained']]
    if restr: ax.annotate('Fijo: '+','.join(restr),(nd['x'],nd['y']),xytext=(0,-30),textcoords='offset points',ha='center',fontsize=9)
ax.set_aspect('equal'); ax.margins(.25); ax.grid(alpha=.2)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_title('Modelo de cálculo (no deformado)')
plt.show()
display(pd.DataFrame([dict(GDL=i+1,**d) for i,d in enumerate(caso['dofs'])]))


## Notación del cálculo

`Fn` representa $F_n$, la carga aplicada en los grados de libertad libres. `Fa` representa $F_a$, la acción nodal neta en los apoyos: suma de la carga aplicada allí y la reacción. Por ejemplo, en 11.7, $F_a=Y_B-20=10$ kN y la reacción es $Y_B=30$ kN.

`FF` y `FFbar` representan $F^F$ y $\bar F^F$. Por elemento, el orden es $[Y_{ij}^F,M_{ij}^F,Y_{ji}^F,M_{ji}^F]$ en vigas; pórticos añaden $X_{ij}^F$ y $X_{ji}^F$. El código conserva `Faplicada` para el registro de cargas conocidas y `Ra`/`R` para calcular reacciones y residuos; no son nuevos vectores de incógnitas del curso.

$w>0$ actúa en $-\bar Y$: hacia abajo en las vigas horizontales. Los momentos positivos son antihorarios. Use kN, m y rad en las operaciones. En la figura original del pórtico, el rótulo $q=25$ hacia abajo equivale a $w=25$ en este cálculo.


## 3. Rigidez local y transformación

Primero calcule $\Delta x=x_j-x_i$, $\Delta y=y_j-y_i$, $L=\sqrt{\Delta x^2+\Delta y^2}$, $c=\Delta x/L$ y $s=\Delta y/L$. El ángulo $\varphi$ se mide antihorario de $X$ a $\bar X$.

Se emplean $\bar\delta_e=T_e\delta_e$ y $K_e=T_e^T\bar K_eT_e$. En la viga, el orden local es $[v_i,\theta_i,v_j,\theta_j]^T$; en el pórtico, $[u_i,v_i,\theta_i,u_j,v_j,\theta_j]^T$.

Para una carga uniforme $w>0$ en $-\bar y$ (hacia abajo en una viga horizontal), las fuerzas de empotramiento son
$$\bar F^F=[wL/2,\ wL^2/12,\ wL/2,\ -wL^2/12]^T.$$
En pórticos se insertan ceros en las componentes axiales. Las acciones de extremo son $\bar F=\bar F^F+\bar K\bar\delta$.

**Compruebe antes de continuar:** ¿qué signo tiene cada componente de $\bar F^F$ para $w>0$? Identifique una entrada de $\bar K$ y sus unidades. `resolver` solo se usa al final como comprobación de sensibilidad; el cálculo principal sigue por secciones.


In [ ]:
"""Cálculo en kN, m y rad. delta representa el vector global δ."""
def propiedades(e, caso):
    parametros={p['key']:p['value'] for p in caso.get('parameters',[])}
    E=parametros[e['E_parameter']] if 'E_parameter' in e else e.get('E',0)
    return e.get('EA',E*e.get('A',0)), e.get('EI',E*e.get('I',0))

def matriz_elemento(caso, e):
    nudos={nd['id']:nd for nd in caso['nodes']}
    ni,nj=nudos[e['i']],nudos[e['j']]
    dx,dy=nj['x']-ni['x'],nj['y']-ni['y']
    L=np.hypot(dx,dy)
    assert L>0, 'Longitud no positiva'
    c,s=dx/L,dy/L
    EA,EI=propiedades(e,caso)
    if caso['kind']=='truss':
        assert EA>0
        Kbar=EA/L*np.array([[1,0,-1,0],[0,0,0,0],[-1,0,1,0],[0,0,0,0]],dtype=float)
        R2=np.array([[c,s],[-s,c]])
        T=np.zeros((4,4));T[:2,:2]=R2;T[2:,2:]=R2
        FFbar=np.zeros(4)
        componentes=['u','v']
    elif caso['kind']=='beam':
        assert EI>0 and abs(s)<1e-12 and c>0
        Kbar=EI/L**3*np.array([[12,6*L,-12,6*L],[6*L,4*L**2,-6*L,2*L**2],[-12,-6*L,12,-6*L],[6*L,2*L**2,-6*L,4*L**2]])
        T=np.eye(4);w=e.get('w',0)
        FFbar=np.array([w*L/2,w*L**2/12,w*L/2,-w*L**2/12])
        componentes=['v','theta']
    elif caso['kind']=='frame':
        assert EA>0 and EI>0
        Kbar=np.array([[EA/L,0,0,-EA/L,0,0],
            [0,12*EI/L**3,6*EI/L**2,0,-12*EI/L**3,6*EI/L**2],
            [0,6*EI/L**2,4*EI/L,0,-6*EI/L**2,2*EI/L],
            [-EA/L,0,0,EA/L,0,0],
            [0,-12*EI/L**3,-6*EI/L**2,0,12*EI/L**3,-6*EI/L**2],
            [0,6*EI/L**2,2*EI/L,0,-6*EI/L**2,4*EI/L]])
        R3=np.array([[c,s,0],[-s,c,0],[0,0,1]])
        T=np.zeros((6,6));T[:3,:3]=R3;T[3:,3:]=R3
        w=e.get('w',0)
        FFbar=np.array([0,w*L/2,w*L**2/12,0,w*L/2,-w*L**2/12])
        componentes=['u','v','theta']
    else:raise ValueError(caso['kind'])
    indices=[next(i for i,d in enumerate(caso['dofs']) if d['node']==nd and d['component']==comp) for nd in [e['i'],e['j']] for comp in componentes]
    return dict(id=e['id'],L=L,c=c,s=s,EA=EA,EI=EI,Kbar=Kbar,T=T,Kglobal=T.T@Kbar@T,FFbar=FFbar,FFglobal=T.T@FFbar,indices=indices)

def carga_aplicada(caso):
    parametros={p['key']:p['value'] for p in caso.get('parameters',[])}
    def valor(d):
        if 'load_expr' not in d:return d.get('load',0.)
        ex=d['load_expr'];return ex.get('scale',1)*np.prod([parametros[k] for k in ex['factors']])
    return np.array([valor(d) for d in caso['dofs']],dtype=float)

def resolver(caso):
    ndof=len(caso['dofs']);K=np.zeros((ndof,ndof));FF=np.zeros(ndof)
    Faplicada=carga_aplicada(caso);elementos=[]
    for e in caso['elements']:
        el=matriz_elemento(caso,e);ids=el['indices'];elementos.append(el)
        K[np.ix_(ids,ids)]+=el['Kglobal'];FF[ids]+=el['FFglobal']
    n=np.array([i for i,d in enumerate(caso['dofs']) if not d['restrained']],dtype=int)
    a=np.array([i for i,d in enumerate(caso['dofs']) if d['restrained']],dtype=int)
    delta=np.zeros(ndof);delta[a]=[caso['dofs'][i].get('prescribed',0.) for i in a]
    da=delta[a];Fn=Faplicada[n]
    Knn=K[np.ix_(n,n)];Kna=K[np.ix_(n,a)];Kan=K[np.ix_(a,n)];Kaa=K[np.ix_(a,a)]
    dn=np.linalg.solve(Knn,Fn-FF[n]-Kna@da);delta[n]=dn
    Fa=FF[a]+Kan@dn+Kaa@da;Ra=Fa-Faplicada[a]
    R=FF+K@delta-Faplicada
    for el in elementos:
        el['delta_global']=delta[el['indices']];el['dbar']=el['T']@el['delta_global']
        el['Fbar']=el['FFbar']+el['Kbar']@el['dbar']
    return dict(K=K,FF=FF,Faplicada=Faplicada,Fn=Fn,Fa=Fa,Ra=Ra,delta=delta,dn=dn,da=da,R=R,Knn=Knn,Kan=Kan,n=n,a=a,elementos=elementos)


In [ ]:
elementos=[matriz_elemento(caso,e) for e in caso['elements']]
for el in elementos:
    print(f"Elemento {el['id']}: L={el['L']:.6g} m; c={el['c']:.6g}; s={el['s']:.6g}; códigos={np.array(el['indices'])+1}")
    for nombre in ['Kbar','T','Kglobal']:
        print(nombre);display(pd.DataFrame(el[nombre]))
    print('F̄^F de empotramiento, orden de extremos i,j:',el['FFbar'])
    print('F^F de empotramiento global:',el['FFglobal'])


## 4. Ensamblaje

Cada $K_e$ se suma en las posiciones señaladas por sus números de código. Los aportes compartidos se suman.
$$K=\sum_e K_e,\qquad F^F=\sum_e T_e^T\bar F_e^F.$$
Las cargas aplicadas se conservan por separado, incluidas las de apoyos. El segundo miembro completo es $F_{\mathrm{aplicada}}-F^F$.

**Compruebe:** escoja una casilla de $K$ compartida por dos elementos y sume sus aportes a mano.


In [ ]:
ndof=len(caso['dofs']);K=np.zeros((ndof,ndof));FF=np.zeros(ndof)
Faplicada=carga_aplicada(caso)
for el in elementos:
    ids=el['indices'];K[np.ix_(ids,ids)]+=el['Kglobal'];FF[ids]+=el['FFglobal']
etiquetas=[d['label'] for d in caso['dofs']]
display(pd.DataFrame(K,index=etiquetas,columns=etiquetas))
display(pd.DataFrame({'Carga nodal (Fn en n)':Faplicada,'F^F empotramiento':FF,'Carga nodal - F^F':Faplicada-FF},index=etiquetas))
fig,ax=plt.subplots(figsize=(7,6));im=ax.imshow(K,cmap='RdBu_r');fig.colorbar(im,ax=ax,label='Rigidez: unidades según pareja de GDL')
ax.set_xticks(range(ndof),etiquetas,rotation=45);ax.set_yticks(range(ndof),etiquetas);ax.set_title('Matriz global K: signo y patrón');plt.show()


## 5. Restricciones y sistema reducido

Identifique $n$ (libres) y $a$ (apoyos). $\delta_a$ es conocido. La ecuación general es
$$K_{nn}\delta_n=F_n-F_n^F-K_{na}\delta_a,\qquad F_n=F_{\mathrm{aplicada},n}.$$
En estos casos $\delta_a=0$. Se resuelve `dn = np.linalg.solve(Knn, Fn - FF[n] - Kna @ da)`. No se invierte la matriz global sin apoyos.

**Compruebe:** ¿qué filas corresponden a desplazamientos desconocidos? Una matriz singular exige revisar apoyos, conectividad, propiedades y unidades.


In [ ]:
n=np.array([i for i,d in enumerate(caso['dofs']) if not d['restrained']],dtype=int)
a=np.array([i for i,d in enumerate(caso['dofs']) if d['restrained']],dtype=int)
delta=np.zeros(ndof);delta[a]=[caso['dofs'][i].get('prescribed',0.) for i in a]
da=delta[a];Fn=Faplicada[n]
Knn=K[np.ix_(n,n)];Kna=K[np.ix_(n,a)];Kan=K[np.ix_(a,n)];Kaa=K[np.ix_(a,a)]
print('GDL n:',n+1,'; GDL a:',a+1)
display(pd.DataFrame(Knn,index=np.array(etiquetas)[n],columns=np.array(etiquetas)[n]))
print('Fn - FF[n] - Kna @ da:',Fn-FF[n]-Kna@da)
assert np.all(np.linalg.eigvalsh(Knn)>0), 'Knn no es definido positivo: revise el modelo.'
print('Condición numérica:',np.linalg.cond(Knn),'(depende de escalas y unidades)')
dn=np.linalg.solve(Knn,Fn-FF[n]-Kna@da);delta[n]=dn


## 5. Calcule acciones de apoyo y fuerzas de extremo

$$F_a=F_a^F+K_{an}\delta_n+K_{aa}\delta_a.$$

La reacción del apoyo es $F_a$ menos la carga aplicada directamente en ese grado de libertad. No omita este último término: en B del ejemplo 11.7 hay una carga de $-20$ kN.

$$\bar\delta_{ij}=T_{ij}\delta_{ij},\qquad \bar F_{ij}=\bar F_{ij}^F+\bar K_{ij}\bar\delta_{ij}.$$

En los grados libres, $F^F+K\delta-F_{\mathrm{aplicada}}$ debe ser casi cero. En los apoyos es la reacción. Si no existe carga aplicada sobre un apoyo, su reacción coincide con $F_a$.


In [ ]:
Fa=FF[a]+Kan@dn+Kaa@da;Ra=Fa-Faplicada[a]
R=FF+K@delta-Faplicada
tabla=pd.DataFrame({'delta':delta,'unidad delta':['rad' if d['component']=='theta' else 'm' for d in caso['dofs']],
    'Reacción en a / residuo en n':R, 'Fn / Fa':FF+K@delta,'unidad fuerza':['kN·m' if d['component']=='theta' else 'kN' for d in caso['dofs']],
    'partición':['a' if d['restrained'] else 'n' for d in caso['dofs']]},index=etiquetas)
display(tabla);print(caso['result_note'])
for linea in caso.get('symbolic_results',[]):print(linea)
for el in elementos:
    el['delta_global']=delta[el['indices']];el['dbar']=el['T']@el['delta_global']
    el['Fbar']=el['FFbar']+el['Kbar']@el['dbar']
    print('Elemento',el['id']);display(pd.DataFrame({'dbar':el['dbar'],'Fbar':el['Fbar']}))
    if caso['kind']=='truss':print('Sij (+ tensión) =',el['Fbar'][2],'kN')


### Deformada y contraste de signos

La línea roja es la deformada amplificada; el factor se muestra. La figura no se usa para medir desplazamientos. En flexión se interpola con funciones de Hermite y se añade la solución particular de la carga uniforme. En la viga solo se representa AB, no el voladizo eliminado. Compruebe si el giro de B es horario y si el pórtico responde de forma simétrica.

In [ ]:
curvas=[]
for el,e in zip(elementos,caso['elements']):
    ni=nudos[e['i']];L=el['L'];c,s=el['c'],el['s'];t=np.linspace(0,1,61);ul=el['dbar']
    if caso['kind']=='truss':
        ua=(1-t)*ul[0]+t*ul[2];va=(1-t)*ul[1]+t*ul[3]
    else:
        h1=1-3*t**2+2*t**3;h2=L*(t-2*t**2+t**3);h3=3*t**2-2*t**3;h4=L*(-t**2+t**3)
        if caso['kind']=='beam':ua=np.zeros_like(t);vv=ul
        else:ua=(1-t)*ul[0]+t*ul[3];vv=ul[[1,2,4,5]]
        va=h1*vv[0]+h2*vv[1]+h3*vv[2]+h4*vv[3]
        va-=e.get('w',0)*L**4*t**2*(1-t)**2/(24*el['EI'])
    x0=ni['x']+c*L*t;y0=ni['y']+s*L*t;dx=c*ua-s*va;dy=s*ua+c*va
    curvas.append((x0,y0,dx,dy))
mov=max(max(np.max(np.abs(x[2])),np.max(np.abs(x[3]))) for x in curvas)
tam=max(max(n['x'] for n in caso['nodes'])-min(n['x'] for n in caso['nodes']),max(n['y'] for n in caso['nodes'])-min(n['y'] for n in caso['nodes']))
factor=.12*tam/mov if mov>0 else 1
fig,ax=plt.subplots(figsize=(9,5))
for x0,y0,dx,dy in curvas:
    ax.plot(x0,y0,'--',color='#8294A6',lw=1)
    ax.plot(x0+factor*dx,y0+factor*dy,color='#BE3D3D',lw=2)
ax.set_aspect('equal');ax.grid(alpha=.2);ax.set_title(f'Deformada amplificada ×{factor:.3g} (rojo); original (discontinuo)')
ax.set_xlabel('x (m)');ax.set_ylabel('y (m)');plt.show()


## 7. Verificación por pasos

Revise la simetría de $K$, el residuo en $n$, el equilibrio global y la energía. Después contraste un desplazamiento, una reacción y una fuerza local con los valores de control del ejemplo original. Si un dato cambió, no compare su resultado con el control de un caso diferente.

**Pregunta:** ¿puede cumplirse el equilibrio aunque se haya modelado un apoyo equivocado? Compruebe también el dibujo, las unidades y las restricciones.


In [ ]:
resultante=np.zeros(3)
for i,d in enumerate(caso['dofs']):
    val=R[i]+Faplicada[i]-FF[i];nd=nudos[d['node']]
    if d['component']=='u':resultante[0]+=val;resultante[2]-=nd['y']*val
    elif d['component']=='v':resultante[1]+=val;resultante[2]+=nd['x']*val
    else:resultante[2]+=val
escala=max(1.,np.max(np.abs(Faplicada-FF)))
assert np.allclose(K,K.T,rtol=1e-12,atol=1e-8), 'K no simétrica.'
assert np.max(np.abs(R[n]))<1e-8*escala, 'Residuo en n excesivo.'
assert np.max(np.abs(resultante))<1e-7*escala, 'Falla equilibrio global.'
energia=.5*delta@K@delta
assert energia>=-1e-10, 'Energía negativa.'
print('Suma X, suma Y, suma M:',resultante)
print('Máx. residuo en n:',np.max(np.abs(R[n])),'; energía:',energia,'kN·m')
# Valores de control: se aplican a los datos originales, no a variantes editadas.
if hashlib.sha256(json.dumps(caso,ensure_ascii=False,sort_keys=True).encode()).hexdigest()==hash_original:
    if caso['id']=='cercha':
        assert np.allclose(delta[n],[-.01,-.11,0],atol=1e-10)
        assert np.allclose(Ra,[10,-50,50],atol=1e-8)
        assert np.allclose([el['Fbar'][2] for el in elementos],[-10,50*np.sqrt(2),0],atol=1e-8)
    elif caso['id']=='viga':
        assert np.isclose(dn[0],-30/6480,atol=1e-12)
        assert np.allclose(Ra,[-10,-15,30],atol=1e-8)
    else:
        assert np.allclose(dn,[.00001822548,-.00013157895,-.00203720337,-.00001822548,-.00013157895,.00203720337],atol=1e-10)
        assert np.allclose(Ra,[17.31420363,75,-17.26226102,-17.31420363,75,17.26226102],atol=1e-7)
    print('Valores de control del ejemplo original comprobados.')
else:print('Datos modificados: contraste con un control independiente para su variante.')


## 8. Sensibilidad y preguntas

Antes de ejecutar, prediga qué ocurre al duplicar todas las cargas. ¿Qué cambia si duplica todas las rigideces? Explique la respuesta con $K_{nn}\delta_n=F_n-F_n^F$. Las pruebas no modifican el caso original.


In [ ]:
doble_carga=copy.deepcopy(caso)
for p in doble_carga.get('parameters',[]):
    if p.get('load_parameter'):p['value']*=2
for d in doble_carga['dofs']:
    if d.get('editable_load'):d['load']*=2
for e in doble_carga['elements']:
    if 'w' in e:e['w']*=2
rc=resolver(doble_carga)
assert np.allclose(rc['delta'],2*delta,rtol=1e-8,atol=1e-10)
assert np.allclose(rc['Ra'],2*Ra,rtol=1e-8,atol=1e-7)
doble_rigidez=copy.deepcopy(caso)
for p in doble_rigidez.get('parameters',[]):
    if p.get('stiffness_parameter'):p['value']*=2
for e in doble_rigidez['elements']:
    if 'E' in e:e['E']*=2
    if 'EI' in e:e['EI']*=2
    if 'EA' in e:e['EA']*=2
rr=resolver(doble_rigidez)
assert np.allclose(rr['delta'],delta/2,rtol=1e-8,atol=1e-10)
assert np.allclose(rr['Ra'],Ra,rtol=1e-8,atol=1e-7)
for base,ec,er in zip(elementos,rc['elementos'],rr['elementos']):
    assert np.allclose(ec['Fbar'],2*base['Fbar'],atol=1e-7)
    assert np.allclose(er['Fbar'],base['Fbar'],atol=1e-7)
print('Sensibilidad comprobada: cargas ×2 duplican delta, Ra y Fbar; rigidez ×2 reduce delta a la mitad.')


## 9. IA como tutor y Python complementario

Puede completar el trabajo con cálculo a mano y Excel. Este notebook es una comprobación opcional, no un requisito.

Pregunta sugerida: «Obtengo una reacción que no cumple el momento global. No me dé la solución. Pídame tres comprobaciones de signos, unidades y números de código y explique qué evidencia confirmaría cada causa».

Registre su predicción, el error localizado y la comprobación que lo resolvió. Contraste una entrada de $K_{nn}$, una reacción de apoyo y una acción $\bar F$ con Excel. No acepte una respuesta solo por coincidir con una salida de IA.

**Límites:** topología fija, pequeñas deformaciones, propiedades lineales positivas y uniones ideales. No se verifican resistencia, segundo orden ni diseño normativo.
